# Agentic SPM Control with an AFM Digital Twin

**IMC-21: Artificial Intelligence Methods for Microscopy Analysis and Knowledge Extraction**

*Final workshop notebook: an LLM-guided microscope-control pipeline for AFM PID tuning and active learning.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pycroscopy/IMC21-Workshop/blob/main/notebooks/15_00_Agentic_SPM_Control_Digital_Twin.ipynb)

---

## Introduction

This notebook turns the AFM digital twin workflow into a small agentic-control exercise. The agent has tools for changing PID parameters, acquiring scans or spectra, computing image-quality metrics, and deciding what to do next.

There are two operating modes:

1. **Real DTMicroscope mode**: connect to the AFM digital twin server from `DTMicroscope` through Pyro on `localhost:9092`.
2. **Mock mode**: use a lightweight local AFM simulator with the same high-level interface. This is the default when the server or data files are unavailable.

The notebook has two parts:

1. **PID tuning**: the agent adjusts the AFM integral gain `I` to improve scan fidelity.
2. **Active learning**: the same agent explores a structure-property map, using Gaussian-process acquisition as a workshop-friendly stand-in for deep kernel learning.

If `XAI_API_KEY` is available, the planner can query Grok. If it is absent, the notebook uses deterministic mock responses so the exercise still runs offline.

<div style="max-width: 80%; border-left: 6px solid #2e7d32; border-top: 1px solid #e0e0e0; border-right: 1px solid #e0e0e0; border-bottom: 1px solid #e0e0e0; border-radius: 4px; padding: 0; margin-bottom: 20px; box-shadow: 0 4px 8px rgba(0,0,0,0.1), 0 1px 3px rgba(0,0,0,0.08); background-color: #ffffff;">
  <div style="background-color: transparent; color: #1b5e20; padding: 10px 15px; font-weight: bold; border-bottom: 1px solid #e0e0e0; display: flex; align-items: center; gap: 8px;">
    <span>Target</span> Learning Goals
  </div>
  <div style="padding: 15px; background-color: transparent; color: #333333;">
  <ul style="margin: 0; padding-left: 20px;">
      <li style="margin-bottom: 8px;">Wrap microscope actions as explicit tools an agent can call.</li>
      <li style="margin-bottom: 8px;">Define objective functions for image quality and scan stability.</li>
      <li style="margin-bottom: 8px;">Compare LLM-guided PID tuning with a simple active-learning optimizer.</li>
      <li style="margin-bottom: 0;">Extend the same agent pattern to structure-property exploration in SPM.</li>
    </ul>
  </div>
</div>

## 0. Setup

The real AFM twin uses `DTMicroscope` and `Pyro5`. The active-learning example uses scikit-learn's Gaussian process tools so it can run in standard workshop environments. The setup cell installs only the light dependencies by default. The optional real-server setup cells below follow the DTMicroscope reference notebooks.

In [ ]:
# --- Setup: Colab-safe. Skip installs if your environment already has these packages. ---
import os
import sys
import subprocess

try:
    import numpy, scipy, sklearn, matplotlib
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'ipywidgets'], check=True)

import os
import json
import time
import math
import textwrap
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({'figure.dpi': 110, 'image.cmap': 'cividis'})
rng = np.random.default_rng(42)

## 1. Optional: start the real AFM digital twin

For Colab or a fresh machine, uncomment and run these cells. They follow the DTMicroscope AFM examples:

```python
!pip install -q pyro5 scifireaders sidpy pynsid
!pip install -q git+https://github.com/pycroscopy/DTMicroscope.git
!run_server_afm
```

The PID notebook uses:

```python
uri = "PYRO:microscope.server@localhost:9092"
mic_server.initialize_microscope("AFM", data_path="dset_spm1.h5")
mic_server.setup_microscope(data_source="Compound_Dataset_1")
mic_server.get_scan(channels=["HeightRetrace"], modification=[{"effect":"real_PID", "kwargs": kwargs}])
```

The DKL notebook uses the same server and adds:

```python
mic_server.go_to(float(px), float(py))
(spec_x, shape_x, dtype_x), (spec_y, shape_y, dtype_y) = mic_server.get_spectrum()
```

In [ ]:
# OPTIONAL REAL-SERVER SETUP. Leave commented unless you want to run the DTMicroscope server here.
# !pip install -q pyro5 scifireaders sidpy pynsid
# !pip install -q git+https://github.com/pycroscopy/DTMicroscope.git
# !run_server_afm

## 2. Define a microscope interface

The agent should not care whether it is talking to a real digital twin or a local mock. We define a small interface with three actions:

- `scan_pid(I, dz, sample_rate)`: acquire an AFM image under PID settings.
- `scan_line(I, coord)`: acquire a single diagnostic line.
- `measure_spectrum(x, y)`: acquire a local spectrum and reduce it to a scalar property.

In [ ]:
def make_spm_surface(n=128, rng=None):
    """Synthetic AFM topography with terraces, particles, and fine texture."""
    rng = np.random.default_rng() if rng is None else rng
    yy, xx = np.mgrid[-1:1:complex(n), -1:1:complex(n)]
    surface = 0.35 * np.tanh(8 * (xx + 0.18*np.sin(3*np.pi*yy)))
    surface += 0.12 * np.sin(9*np.pi*xx + 2*np.pi*yy)
    for cx, cy, amp, sig in [(-0.45, 0.35, 0.55, 0.09), (0.18, -0.2, 0.45, 0.13), (0.55, 0.45, 0.35, 0.08)]:
        surface += amp * np.exp(-((xx-cx)**2 + (yy-cy)**2)/(2*sig**2))
    surface += 0.03 * rng.normal(size=(n, n))
    surface = gaussian_filter(surface, 0.7)
    surface = (surface - surface.min()) / (surface.max() - surface.min())
    return surface


def simulate_pid_scan(surface, I=10.0, dz=5e-9, sample_rate=2000, trace='forward', direction='horizontal', rng=None):
    """Approximate PID artifacts: low I lags, high I rings, very high I adds instability."""
    rng = np.random.default_rng() if rng is None else rng
    img = surface.copy()
    if direction == 'vertical':
        img = img.T
    if trace == 'backward':
        img = img[:, ::-1]

    optimum = 10.0
    alpha = np.clip(I / (I + 4.0), 0.03, 0.98)       # feedback tracking strength
    ring = max(0.0, (I - 13.0) / 20.0)               # overshoot for aggressive I
    laggy = np.zeros_like(img)
    for row in range(img.shape[0]):
        y = img[row, 0]
        v = 0.0
        for col in range(img.shape[1]):
            err = img[row, col] - y
            v = 0.55*v + alpha*err
            y = y + v + ring*np.sin(0.7*col)*err
            laggy[row, col] = y

    if trace == 'backward':
        laggy = laggy[:, ::-1]
    if direction == 'vertical':
        laggy = laggy.T

    noise = 0.012 + 0.0015*abs(I - optimum) + 0.010*max(0, I - 20) / 20
    laggy += rng.normal(0, noise, laggy.shape)
    laggy = np.clip(laggy, -0.1, 1.2)
    return laggy


@dataclass
class ScanResult:
    I: float
    dz: float
    sample_rate: float
    score: float
    sharpness: float
    trace_mismatch: float
    noise: float
    saturation: float
    image: np.ndarray
    retrace: np.ndarray


class MockAFMDigitalTwin:
    """Local stand-in for DTMicroscope's AFM server."""
    def __init__(self, n=128, rng=None):
        self.rng = np.random.default_rng() if rng is None else rng
        self.surface = make_spm_surface(n, self.rng)
        self.x = 0.0
        self.y = 0.0
        yy, xx = np.mgrid[0:n, 0:n]
        self.property_map = (0.7*np.exp(-((xx-35)**2+(yy-88)**2)/(2*16**2)) +
                             1.0*np.exp(-((xx-92)**2+(yy-43)**2)/(2*20**2)) +
                             0.15*np.sin(xx/8)*np.cos(yy/9))
        self.property_map = (self.property_map - self.property_map.min()) / np.ptp(self.property_map)

    def scan_pid(self, I=10.0, dz=5e-9, sample_rate=2000):
        forward = simulate_pid_scan(self.surface, I, dz, sample_rate, 'forward', 'horizontal', self.rng)
        backward = simulate_pid_scan(self.surface, I, dz, sample_rate, 'backward', 'horizontal', self.rng)
        return forward, backward

    def scan_line(self, I=10.0, coord=-1e-6, direction='vertical'):
        img, _ = self.scan_pid(I=I)
        idx = int(np.clip((coord + 1e-6)/(2e-6) * (img.shape[0]-1), 0, img.shape[0]-1))
        return img[:, idx] if direction == 'vertical' else img[idx, :]

    def go_to(self, x, y):
        self.x, self.y = float(x), float(y)

    def measure_spectrum(self, x, y):
        self.go_to(x, y)
        n = self.property_map.shape[0]
        px = int(np.clip(round(x*(n-1)), 0, n-1))
        py = int(np.clip(round(y*(n-1)), 0, n-1))
        value = self.property_map[py, px]
        voltage = np.linspace(-4, 4, 128)
        spectrum = value*np.tanh(1.6*voltage) + 0.12*np.sin(2*np.pi*voltage) + self.rng.normal(0, 0.015, voltage.size)
        return voltage, spectrum, float(value)


class DTMicroscopeAFMClient:
    """Thin adapter for the real DTMicroscope Pyro AFM server."""
    def __init__(self, uri='PYRO:microscope.server@localhost:9092', data_path='dset_spm1.h5', data_source='Compound_Dataset_1'):
        import Pyro5.api
        self.mic_server = Pyro5.api.Proxy(uri)
        self.mic_server.initialize_microscope('AFM', data_path=data_path)
        self.mic_server.setup_microscope(data_source=data_source)
        self.x = getattr(self.mic_server, 'x', 0.0)
        self.y = getattr(self.mic_server, 'y', 0.0)

    def _reshape(self, response):
        array_list, shape, dtype = response
        return np.array(array_list, dtype=dtype).reshape(shape)

    def scan_pid(self, I=10.0, dz=5e-9, sample_rate=2000):
        kwargs = {'I': float(I), 'dz': float(dz), 'sample_rate': int(sample_rate)}
        mod = [{'effect': 'real_PID', 'kwargs': kwargs}]
        fwd = self._reshape(self.mic_server.get_scan(channels=['HeightRetrace'], modification=mod,
                                                     trace='forward', direction='horizontal'))[0].T
        bwd = self._reshape(self.mic_server.get_scan(channels=['HeightRetrace'], modification=mod,
                                                     trace='backward', direction='horizontal'))[0].T
        return fwd, bwd

    def scan_line(self, I=10.0, coord=-1e-6, direction='vertical'):
        kwargs = {'I': float(I), 'dz': 5e-9, 'sample_rate': 2000}
        mod = [{'effect': 'real_PID', 'kwargs': kwargs}]
        line = self._reshape(self.mic_server.scan_individual_line(direction, channels=['HeightRetrace'],
                                                                  coord=float(coord), modification=mod,
                                                                  trace='forward'))
        return np.squeeze(line)

    def go_to(self, x, y):
        self.mic_server.go_to(float(x), float(y))
        self.x, self.y = float(x), float(y)

    def measure_spectrum(self, x, y):
        self.go_to(x, y)
        (spec_x, shape_x, dtype_x), (spec_y, shape_y, dtype_y) = self.mic_server.get_spectrum()
        voltage = np.array(spec_x, dtype=dtype_x).reshape(shape_x)
        spectrum = np.array(spec_y, dtype=dtype_y).reshape(shape_y)
        return voltage, spectrum, loop_area(spectrum, cycle=8)


def connect_afm(use_real_server=False, **kwargs):
    if use_real_server:
        try:
            afm = DTMicroscopeAFMClient(**kwargs)
            print('Connected to real DTMicroscope AFM server.')
            return afm
        except Exception as exc:
            print('Real AFM server unavailable; falling back to mock twin:', repr(exc))
    print('Using local mock AFM digital twin.')
    return MockAFMDigitalTwin(rng=rng)

afmscope = connect_afm(use_real_server=False)

## 3. Image-quality objective for PID tuning

An autonomous microscope needs an objective. Here we combine four diagnostics:

- **sharpness**: image gradients should be strong for resolved features.
- **trace mismatch**: forward and backward scans should agree.
- **noise**: high-frequency residual should be modest.
- **saturation**: unstable scans should not clip or overshoot.

The score is deliberately visible and editable. In real projects, this is where domain knowledge enters.

In [ ]:
def robust_normalize(img):
    lo, hi = np.percentile(img, [1, 99])
    return np.clip((img - lo) / (hi - lo + 1e-12), 0, 1)


def image_quality(forward, retrace):
    f = robust_normalize(np.asarray(forward, dtype=float))
    r = robust_normalize(np.asarray(retrace, dtype=float))
    gy, gx = np.gradient(f)
    sharpness = float(np.mean(np.sqrt(gx**2 + gy**2)))
    trace_mismatch = float(np.mean(np.abs(f - r)))
    smooth = gaussian_filter(f, 1.0)
    noise = float(np.std(f - smooth))
    saturation = float(np.mean((forward <= np.percentile(forward, 0.5)) | (forward >= np.percentile(forward, 99.5))))
    score = 3.0*sharpness - 1.7*trace_mismatch - 1.2*noise - 0.5*saturation
    return {'score': score, 'sharpness': sharpness, 'trace_mismatch': trace_mismatch,
            'noise': noise, 'saturation': saturation}


def evaluate_pid(afm, I, dz=5e-9, sample_rate=2000, show=False):
    forward, retrace = afm.scan_pid(I=I, dz=dz, sample_rate=sample_rate)
    metrics = image_quality(forward, retrace)
    result = ScanResult(I=float(I), dz=float(dz), sample_rate=float(sample_rate),
                        image=forward, retrace=retrace, **metrics)
    if show:
        fig, ax = plt.subplots(1, 3, figsize=(10, 3.2))
        ax[0].imshow(forward, origin='lower'); ax[0].set_title(f'forward, I={I:.2f}')
        ax[1].imshow(retrace, origin='lower'); ax[1].set_title('retrace')
        ax[2].imshow(np.abs(robust_normalize(forward)-robust_normalize(retrace)), origin='lower', cmap='magma')
        ax[2].set_title(f'|difference|, score={result.score:.3f}')
        for a in ax: a.set_xticks([]); a.set_yticks([])
        plt.tight_layout()
    return result

# Compare classic bad/good PID examples from the reference notebook.
trial_Is = [0.5, 5, 10, 20, 30]
trial_results = [evaluate_pid(afmscope, I) for I in trial_Is]
for r in trial_results:
    print(f'I={r.I:5.1f} | score={r.score: .4f} | sharp={r.sharpness:.4f} | mismatch={r.trace_mismatch:.4f} | noise={r.noise:.4f}')

fig, ax = plt.subplots(1, len(trial_results), figsize=(13, 3))
for a, res in zip(ax, trial_results):
    a.imshow(res.image, origin='lower')
    a.set_title(f'I={res.I:g}\nscore={res.score:.3f}')
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

## 4. A minimal LLM microscope-control agent

The LLM is not allowed to directly operate the microscope. Instead, it proposes a JSON action, and Python validates bounds before calling a tool. This keeps the control loop inspectable:

```json
{"action": "evaluate_pid", "I": 8.0, "reason": "try near the best previous score"}
```

If `XAI_API_KEY` is set, the planner tries to call Grok using an OpenAI-compatible client. If not, it uses a simple heuristic that is intentionally easy for students to beat.

In [ ]:
class GrokOrMockPlanner:
    def __init__(self, model='grok-4-fast-reasoning', temperature=0.2):
        self.model = model
        self.temperature = temperature
        self.api_key = os.getenv('XAI_API_KEY') or os.getenv('GROK_API_KEY')
        self.client = None
        if self.api_key:
            try:
                from openai import OpenAI
                self.client = OpenAI(api_key=self.api_key, base_url=os.getenv('XAI_BASE_URL', 'https://api.x.ai/v1'))
                print('Grok planner enabled.')
            except Exception as exc:
                print('Could not initialize Grok client; using mock planner:', repr(exc))
                self.client = None
        else:
            print('No XAI_API_KEY/GROK_API_KEY found; using mock planner.')

    def _heuristic(self, history, bounds):
        low, high = bounds
        tried = np.array([h['I'] for h in history], dtype=float)
        scores = np.array([h['score'] for h in history], dtype=float)
        if len(history) < 3:
            proposal = [low, (low+high)/2, high][len(history)]
        else:
            best_i = int(np.argmax(scores))
            best = tried[best_i]
            span = max(0.4, (high-low) / (2 + len(history)))
            candidates = np.array([best - span, best + span, best - 0.5*span, best + 0.5*span])
            candidates = np.clip(candidates, low, high)
            # Prefer untried candidates, otherwise shrink around the best.
            proposal = candidates[np.argmax([np.min(np.abs(tried - c)) for c in candidates])]
        return {'action': 'evaluate_pid', 'I': float(proposal), 'reason': 'mock planner explores around the best score so far'}

    def propose_pid(self, history, bounds=(0.2, 35.0)):
        if self.client is None:
            return self._heuristic(history, bounds)
        prompt = f"""
You are controlling an AFM digital twin. Propose exactly one next PID integral gain I.
Return only JSON with keys action, I, reason. action must be evaluate_pid.
Bounds: {bounds}. History: {json.dumps(history[-10:], indent=2)}
Optimize score. High trace_mismatch/noise/saturation are bad; high sharpness is good.
"""
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                temperature=self.temperature,
                messages=[{'role': 'user', 'content': prompt}],
            )
            text = response.choices[0].message.content.strip()
            data = json.loads(text[text.find('{'):text.rfind('}')+1])
            data['I'] = float(np.clip(float(data['I']), bounds[0], bounds[1]))
            data['action'] = 'evaluate_pid'
            return data
        except Exception as exc:
            print('Planner call failed; using mock proposal:', repr(exc))
            return self._heuristic(history, bounds)


class PIDTuningAgent:
    def __init__(self, afm, planner=None, bounds=(0.2, 35.0)):
        self.afm = afm
        self.planner = planner or GrokOrMockPlanner()
        self.bounds = bounds
        self.history = []

    def observe(self, result, reason='initial'):
        row = {k: getattr(result, k) for k in ['I', 'score', 'sharpness', 'trace_mismatch', 'noise', 'saturation']}
        row['reason'] = reason
        self.history.append(row)

    def step(self, show=False):
        proposal = self.planner.propose_pid(self.history, self.bounds)
        I = float(np.clip(proposal.get('I', 10.0), *self.bounds))
        result = evaluate_pid(self.afm, I, show=show)
        self.observe(result, proposal.get('reason', ''))
        return proposal, result

    def best(self):
        return max(self.history, key=lambda x: x['score'])

agent = PIDTuningAgent(afmscope)

# Seed the agent with two measurements so it has a starting context.
for I in [2.0, 20.0]:
    agent.observe(evaluate_pid(afmscope, I), reason='seed measurement')

for _ in range(8):
    proposal, result = agent.step(show=False)
    print(f"I={result.I:6.2f} score={result.score: .4f} | {proposal['reason'][:80]}")

print('Best:', agent.best())

In [ ]:
# Plot the agent's PID tuning trajectory.
history = agent.history
Is = np.array([h['I'] for h in history])
scores = np.array([h['score'] for h in history])
best = agent.best()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(Is, scores, 'o-')
ax[0].scatter([best['I']], [best['score']], s=100, marker='x', c='k')
ax[0].set_xlabel('I gain')
ax[0].set_ylabel('image-quality score')
ax[0].set_title('agent PID search')
ax[1].plot(scores, 'o-')
ax[1].set_xlabel('agent step')
ax[1].set_ylabel('score')
ax[1].set_title('score over time')
plt.tight_layout()

best_scan = evaluate_pid(afmscope, best['I'], show=True)

## 5. Active learning with the same agent pattern

The DKL reference notebook extracts local image patches, measures spectra at selected positions, converts each spectrum to a scalar descriptor such as hysteresis-loop area, then uses an acquisition function to choose the next measurement.

Here we keep the same microscope-control pattern but use a Gaussian process on coordinates plus local image features. This is fast and readable. Students can replace `fit_gp_and_acquire` with GPax DKL if they install the full stack.

In [ ]:
def loop_area(raw_spec, cycle=8):
    raw_spec = np.asarray(raw_spec).ravel()
    raw_spec_len = len(raw_spec)
    cycle_len = int(raw_spec_len / cycle)
    half_len = int(cycle_len / 2)
    q_len = int(cycle_len / 4)
    if cycle_len < 4:
        return float(np.ptp(raw_spec))
    loop_top = [raw_spec[q_len:q_len+half_len], raw_spec[q_len+2*half_len:q_len+3*half_len],
                raw_spec[q_len+4*half_len:2*q_len+4*half_len]]
    loop_bottom = [raw_spec[:q_len], raw_spec[q_len+half_len:q_len+2*half_len],
                   raw_spec[q_len+3*half_len:q_len+4*half_len]]
    return float(abs(np.sum(np.concatenate(loop_top)) - np.sum(np.concatenate(loop_bottom))))


def make_candidate_features(image, grid_n=24, patch_radius=3):
    """Coordinate + local image statistics features for active learning."""
    img = robust_normalize(image)
    n = img.shape[0]
    ys = np.linspace(patch_radius, n-patch_radius-1, grid_n).astype(int)
    xs = np.linspace(patch_radius, n-patch_radius-1, grid_n).astype(int)
    rows, coords = [], []
    gy, gx = np.gradient(img)
    for y in ys:
        for x in xs:
            patch = img[y-patch_radius:y+patch_radius+1, x-patch_radius:x+patch_radius+1]
            gpatch = np.sqrt(gx[y-patch_radius:y+patch_radius+1, x-patch_radius:x+patch_radius+1]**2 +
                             gy[y-patch_radius:y+patch_radius+1, x-patch_radius:x+patch_radius+1]**2)
            coords.append([x/(n-1), y/(n-1)])
            rows.append([x/(n-1), y/(n-1), patch.mean(), patch.std(), gpatch.mean()])
    return np.array(coords), np.array(rows)


def expected_improvement(mu, sigma, best_y, xi=0.01):
    sigma = np.maximum(sigma, 1e-9)
    imp = mu - best_y - xi
    z = imp / sigma
    return imp * norm.cdf(z) + sigma * norm.pdf(z)


def fit_gp_and_acquire(X_measured, y_measured, X_candidates, candidate_mask):
    scaler = StandardScaler().fit(X_measured)
    Xm = scaler.transform(X_measured)
    Xc = scaler.transform(X_candidates)
    kernel = ConstantKernel(1.0, (0.1, 10.0)) * Matern(length_scale=np.ones(Xm.shape[1]), nu=2.5) + WhiteKernel(1e-4)
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=0, n_restarts_optimizer=2)
    gp.fit(Xm, y_measured)
    mu, sigma = gp.predict(Xc, return_std=True)
    acq = expected_improvement(mu, sigma, np.max(y_measured))
    acq[~candidate_mask] = -np.inf
    next_idx = int(np.argmax(acq))
    return gp, mu, sigma, acq, next_idx

# Use the tuned scan as the structural image for active learning.
structure_image = best_scan.image
coords, features = make_candidate_features(structure_image, grid_n=24)
measured_mask = np.zeros(len(coords), dtype=bool)

# Seed with a few random measurements.
seed_idx = rng.choice(len(coords), size=5, replace=False)
measurements = []
for idx in seed_idx:
    x, y = coords[idx]
    voltage, spectrum, scalar = afmscope.measure_spectrum(float(x), float(y))
    measurements.append({'idx': int(idx), 'x': float(x), 'y': float(y), 'scalar': float(scalar)})
    measured_mask[idx] = True

print(f'Initial measurements: {len(measurements)}')

In [ ]:
# Run active learning.
active_steps = 18
for step in range(active_steps):
    measured_idx = np.array([m['idx'] for m in measurements], dtype=int)
    y_measured = np.array([m['scalar'] for m in measurements], dtype=float)
    candidate_mask = ~measured_mask
    gp, mu, sigma, acq, next_idx = fit_gp_and_acquire(features[measured_idx], y_measured, features, candidate_mask)
    x, y = coords[next_idx]
    voltage, spectrum, scalar = afmscope.measure_spectrum(float(x), float(y))
    measurements.append({'idx': int(next_idx), 'x': float(x), 'y': float(y), 'scalar': float(scalar)})
    measured_mask[next_idx] = True
    print(f'{step+1:02d}/{active_steps}: measured ({x:.2f}, {y:.2f}) scalar={scalar:.3f}, best={max(m["scalar"] for m in measurements):.3f}')

measured_idx = np.array([m['idx'] for m in measurements], dtype=int)
y_measured = np.array([m['scalar'] for m in measurements], dtype=float)
gp, mu, sigma, acq, next_idx = fit_gp_and_acquire(features[measured_idx], y_measured, features, ~measured_mask)

In [ ]:
# Visualize the active-learning run.
grid_n = int(np.sqrt(len(coords)))
mu_img = mu.reshape(grid_n, grid_n)
sigma_img = sigma.reshape(grid_n, grid_n)
acq_img = np.where(np.isfinite(acq), acq, np.nan).reshape(grid_n, grid_n)
pts = np.array([[m['x'], m['y']] for m in measurements])
vals = np.array([m['scalar'] for m in measurements])

fig, ax = plt.subplots(1, 4, figsize=(14, 3.6))
ax[0].imshow(structure_image, origin='lower')
ax[0].scatter(pts[:,0]*(structure_image.shape[1]-1), pts[:,1]*(structure_image.shape[0]-1), c=vals, s=26, cmap='jet', edgecolor='k')
ax[0].set_title('measured locations')
ax[1].imshow(mu_img, origin='lower', cmap='viridis')
ax[1].set_title('GP predicted property')
ax[2].imshow(sigma_img, origin='lower', cmap='magma')
ax[2].set_title('uncertainty')
ax[3].imshow(acq_img, origin='lower', cmap='plasma')
ax[3].set_title('expected improvement')
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

plt.figure(figsize=(5, 3.2))
plt.plot(np.maximum.accumulate(vals), 'o-')
plt.xlabel('measurement number')
plt.ylabel('best scalar found')
plt.title('active-learning progress')
plt.tight_layout()

## 6. Exercises

1. **Change the objective.** Edit `image_quality`. Make the score care more about trace-retrace mismatch and less about sharpness. Does the agent choose a different `I`?
2. **Constrain the instrument.** Change `bounds` in `PIDTuningAgent` to `(1, 15)`. What happens if the true optimum is near the edge of the allowed range?
3. **Compare planners.** Replace the mock planner with a grid search or Bayesian optimizer over `I`. Which uses fewer scans?
4. **Prompt engineering.** Set `XAI_API_KEY` and modify the planner prompt so Grok must explain whether it is exploring or exploiting. Does the behavior change?
5. **Bad metric failure.** Remove the trace-mismatch penalty. Can the agent be fooled by high-gain scans that look sharp but are unstable?
6. **Real DTMicroscope run.** Start `run_server_afm`, set `use_real_server=True`, and run the first PID section. Compare the best `I` with the reference notebook's low, normal, and high examples.
7. **DKL upgrade.** Replace `fit_gp_and_acquire` with the GPax `viDKL` loop from the DKL reference notebook. Keep the agent tool boundary the same: choose point, call `go_to`, call `get_spectrum`, update the dataset.
8. **Human-in-the-loop control.** Add a rule that the agent must ask for confirmation before trying `I > 25` or before measuring outside a user-selected region of interest.
9. **Multi-objective control.** Tune both `I` and `sample_rate`. Define a score that also penalizes acquisition time.
10. **Scientific target.** Change the active-learning scalar from `loop_area` to coercive voltage, remanent response, or spectral hysteresis asymmetry.